# 02 · Feature Extraction

Notebook 01 produced three kinds of *prepared* artifacts: clinical embeddings (already final —
nothing more to do with those), WSI tissue-patch coordinates, and the label tables. This notebook
turns the WSI patch coordinates and the raw MRI scans into model embeddings:

- **WSI → [H-optimus-0](https://huggingface.co/bioptimus/H-optimus-0)**: a ViT pathology
  foundation model, run over each tissue patch from notebook 01's manifests. 1536-dim embedding
  per patch, CLS token from the *last* transformer block only (no layer-wise aggregation).
- **MRI → MRI-PTPCa**: a CNN+ViT model adapted from HIMF-Surv. Unlike WSI, MRI preprocessing
  (resampling, mask-cropping, histogram equalization, normalization) is *not* a separate step —
  it happens right here, inline, before the model runs. 2048-dim embedding per scan, again from
  the last transformer block only.

**Where the code lives:** H-optimus-0 loading is 3 lines via `timm`, so it stays inline below.
MRI-PTPCa's architecture (`CNNViTMM`, `VisionNet`) is a large, adapted third-party model with
input-reshaping plumbing that isn't itself the lesson — it lives in `src/mri_ptpca.py`. This
notebook still writes and explains the *preprocessing* function inline, since that's the part
worth actually reading.

**Prerequisites:** notebook 00 (environment + kernel), and a HuggingFace token in `.env` with
access granted to H-optimus-0 (also notebook 00, Step 5) — H-optimus-0 is a gated model.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> repo root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
import numpy as np
import torch
import timm
import matplotlib.pyplot as plt
import openslide
import SimpleITK as sitk
from PIL import Image
from dotenv import load_dotenv
from timm.data import create_transform, resolve_data_config
from torch.nn import functional as F
from tqdm.auto import tqdm

from src.mri_ptpca import EMBEDDING_DIM as MRI_EMBEDDING_DIM
from src.mri_ptpca import extract_embedding as extract_mri_embedding
from src.mri_ptpca import load_mri_ptpca_model

Image.MAX_IMAGE_PIXELS = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
# --- Project paths -----------------------------------------------------------
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "task1"
WSI_IMAGE_DIR = RAW_DATA_DIR / "pathology" / "images"
MRI_IMAGE_DIR = RAW_DATA_DIR / "radiology" / "images"

PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"
WSI_PATCH_MANIFEST_DIR = PREPARED_DIR / "wsi_patch_manifests"  # from notebook 01

FEATURES_DIR = PROJECT_ROOT / "data" / "features"
WSI_FEATURE_DIR = FEATURES_DIR / "wsi"
MRI_FEATURE_DIR = FEATURES_DIR / "mri"

for d in (WSI_FEATURE_DIR, MRI_FEATURE_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError(
        "HF_TOKEN not set. Copy .env.example to .env, fill in a token from "
        "https://huggingface.co/settings/tokens, and make sure your account has access to "
        "https://huggingface.co/bioptimus/H-optimus-0 -- see notebook 00, Step 5."
    )

from huggingface_hub import login
login(token=hf_token)
print("Logged in to HuggingFace.")

## Step 1 · WSI patch embeddings (H-optimus-0)

H-optimus-0 is a standard ViT: 40 blocks, 1536-dim tokens, 224×224 input patches at 0.5
microns/pixel. We register a forward hook on the *last* block only, run each batch of patches
through the model, and take the CLS token (index 0) from that hook — same convention HIMF-Surv
uses for WSI (as opposed to the mean-token pooling MRI-PTPCa's ViT uses, which has no CLS token).

In [ ]:
wsi_model = timm.create_model(
    "hf-hub:bioptimus/H-optimus-0",
    pretrained=True,
    init_values=1e-5,
    dynamic_img_size=False,
)
wsi_model = wsi_model.to(DEVICE).eval()

wsi_transform = create_transform(**resolve_data_config({}, model=wsi_model))
print(wsi_transform)

In [ ]:
def extract_patch_embeddings(wsi_path, coords, patch_size, patch_level, model, transform, device, batch_size=256):
    """Run H-optimus-0 over every (x, y) patch coordinate and return an (N, 1536) array of
    last-block CLS token embeddings."""
    slide = openslide.OpenSlide(str(wsi_path))

    captured = {}
    def hook_fn(module, inp, output):
        captured["last_block"] = output.detach()
    handle = model.blocks[-1].register_forward_hook(hook_fn)

    embeddings = []
    try:
        for i in range(0, len(coords), batch_size):
            batch_coords = coords[i : i + batch_size]
            patches = [
                slide.read_region((int(x), int(y)), patch_level, (patch_size, patch_size)).convert("RGB")
                for x, y in batch_coords
            ]
            batch = torch.stack([transform(p) for p in patches]).to(device)

            with torch.no_grad():
                model(batch)

            cls_tokens = captured["last_block"][:, 0, :]  # (B, 1536)
            embeddings.append(cls_tokens.cpu().numpy())
    finally:
        handle.remove()
        slide.close()

    return np.concatenate(embeddings, axis=0)

Try it on one WSI manifest from notebook 01 first:

In [ ]:
manifest_paths = sorted(WSI_PATCH_MANIFEST_DIR.glob("*_patches.npz"))
if not manifest_paths:
    raise FileNotFoundError(
        f"No patch manifests found in {WSI_PATCH_MANIFEST_DIR} -- run notebook 01 (Step 4) first."
    )

sample_manifest = np.load(manifest_paths[0])
sample_wsi_id = manifest_paths[0].stem.removesuffix("_patches")
sample_patient_id = sample_wsi_id.split("_")[0]
sample_wsi_path = WSI_IMAGE_DIR / sample_patient_id / f"{sample_wsi_id}.tif"

sample_embeddings = extract_patch_embeddings(
    sample_wsi_path,
    coords=sample_manifest["coords"],
    patch_size=int(sample_manifest["patch_size"]),
    patch_level=int(sample_manifest["patch_level"]),
    model=wsi_model,
    transform=wsi_transform,
    device=DEVICE,
)
print(f"{sample_wsi_id}: {sample_embeddings.shape}")

Now run it over every WSI with a saved patch manifest:

In [ ]:
for manifest_path in tqdm(manifest_paths, desc="WSI feature extraction"):
    wsi_id = manifest_path.stem.removesuffix("_patches")
    out_path = WSI_FEATURE_DIR / f"{wsi_id}_features.npy"
    if out_path.exists():
        continue  # already extracted

    patient_id = wsi_id.split("_")[0]
    wsi_path = WSI_IMAGE_DIR / patient_id / f"{wsi_id}.tif"
    manifest = np.load(manifest_path)

    try:
        embeddings = extract_patch_embeddings(
            wsi_path,
            coords=manifest["coords"],
            patch_size=int(manifest["patch_size"]),
            patch_level=int(manifest["patch_level"]),
            model=wsi_model,
            transform=wsi_transform,
            device=DEVICE,
        )
        np.save(out_path, embeddings)
    except Exception as e:
        print(f"Failed on {wsi_id}: {e}")

print(f"WSI features saved to {WSI_FEATURE_DIR}")

## Step 2 · MRI embeddings (MRI-PTPCa)

MRI-PTPCa expects three modalities (T2W, ADC, and a high-b-value/DWI series). HIMF-Surv's
`feature_extractors/mri.py` only ever loads T2W and ADC and zero-fills the third input -- but
CHIMERA actually ships a matching `_hbv.mha` (high b-value) series per scan, so this course uses
it as the real DWI input instead of throwing it away (`src/mri_ptpca.extract_embedding` falls back
to zero-filling only if a scan is missing its HBV file).

The preprocessing pipeline below mirrors HIMF-Surv's `feature_extractors/mri.py` step for step,
applied to all three series:

1. Load the `.mha` volume and its lesion/gland mask with SimpleITK
2. Resize the mask to match the volume if needed, then crop both to the mask's bounding box
3. Resize the cropped volume to a fixed `(16, 200, 200)` (slices, H, W) via trilinear interpolation
4. Per-slice histogram equalization (`histogram_balance`) so slice contrast is comparable across
   scanners/patients
5. Rescale to `[-1, 1]`, then z-score normalize -- both computed over the mask region only, if a
   mask is available
6. Zero out everything outside the mask, then replicate the single channel to 3 (the CNN backbone
   expects RGB-shaped input)

In [ ]:
def histogram_balance(mri_image: np.ndarray, gray_level: int = 65535) -> np.ndarray:
    """Per-slice histogram equalization (assumes 16-bit-range intensities)."""
    equalized = np.zeros_like(mri_image)
    for i in range(mri_image.shape[0]):
        slice_img = np.clip(mri_image[i], 0, gray_level).astype(np.uint16)
        hist, _ = np.histogram(slice_img.ravel(), gray_level, [0, gray_level])
        cdf = np.cumsum(hist)
        cdf_m = np.ma.masked_equal(cdf, 0)
        cdf_m = (cdf_m - cdf_m.min()) * gray_level / (cdf_m.max() - cdf_m.min())
        cdf = np.ma.filled(cdf_m, 0).astype("uint16")
        equalized[i] = cdf[slice_img]
    return equalized


def preprocess_mri_scan(mri_path, mask_path=None, img_size=(16, 200, 200)) -> torch.Tensor:
    """Load one MRI series (.mha) and return a preprocessed (1, D, 3, H, W) tensor, ready for
    MRI-PTPCa. See the step list above for what each stage does."""
    data = sitk.GetArrayFromImage(sitk.ReadImage(str(mri_path))).astype(np.float32)

    mask_resized = None
    if mask_path is not None and Path(mask_path).exists():
        mask_data = sitk.GetArrayFromImage(sitk.ReadImage(str(mask_path)))

        if mask_data.shape != data.shape:
            mask_t = torch.from_numpy(mask_data).float()[None, None]
            mask_data = F.interpolate(mask_t, size=data.shape, mode="nearest").squeeze().numpy()

        nonzero = np.nonzero(mask_data > 0)
        if len(nonzero[0]) > 0:
            (z0, z1), (y0, y1), (x0, x1) = [(a.min(), a.max() + 1) for a in nonzero]
            data = data[z0:z1, y0:y1, x0:x1]
            mask_cropped = mask_data[z0:z1, y0:y1, x0:x1]

            mask_t = torch.from_numpy(mask_cropped).float()[None, None]
            if tuple(mask_t.shape[2:]) != img_size:
                mask_t = F.interpolate(mask_t, size=img_size, mode="nearest")
            mask_resized = mask_t.squeeze().numpy() > 0

    data_t = torch.from_numpy(data).float()[None]  # (1, D, H, W)
    if tuple(data_t.shape[1:]) != img_size:
        data_t = F.interpolate(data_t[None], size=img_size, mode="trilinear", align_corners=False).squeeze(0)
    data = data_t.squeeze(0).numpy()

    data = histogram_balance(data, gray_level=65535).astype(np.float32)

    ref = data[mask_resized] if mask_resized is not None else data
    data_min, data_max = ref.min(), ref.max()
    data = 2.0 * (data - data_min) / (data_max - data_min) - 1.0 if data_max - data_min > 1e-8 else np.zeros_like(data)

    ref = data[mask_resized] if mask_resized is not None else data
    mean, std = ref.mean(), ref.std()
    if std > 1e-8:
        data = (data - mean) / std

    if mask_resized is not None:
        data = data * mask_resized

    tensor = torch.from_numpy(data).float().unsqueeze(1).repeat(1, 3, 1, 1)  # (D, H, W) -> (D, 3, H, W)
    return tensor.unsqueeze(0)  # -> (1, D, 3, H, W)

Try it on one patient, and look at a middle slice before vs. after preprocessing:

In [ ]:
mri_patient_dirs = sorted(d for d in MRI_IMAGE_DIR.iterdir() if d.is_dir())
if not mri_patient_dirs:
    raise FileNotFoundError(f"No patient folders found under {MRI_IMAGE_DIR}")

sample_mri_patient_dir = mri_patient_dirs[0]
sample_t2_path = sorted(sample_mri_patient_dir.glob("*_t2w.mha"))[0]
sample_mri_id = sample_t2_path.stem.removesuffix("_t2w")  # e.g. "1003_0001"
sample_adc_path = sample_mri_patient_dir / f"{sample_mri_id}_adc.mha"
sample_hbv_path = sample_mri_patient_dir / f"{sample_mri_id}_hbv.mha"
sample_mask_path = sample_mri_patient_dir / f"{sample_mri_id}_mask.mha"

raw_t2 = sitk.GetArrayFromImage(sitk.ReadImage(str(sample_t2_path)))
t2_tensor = preprocess_mri_scan(sample_t2_path, sample_mask_path)
print(f"{sample_mri_id}: raw {raw_t2.shape} -> preprocessed {tuple(t2_tensor.shape)}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(raw_t2[raw_t2.shape[0] // 2], cmap="gray")
axes[0].set_title("Raw T2W (middle slice)")
axes[1].imshow(t2_tensor[0, t2_tensor.shape[1] // 2, 0].numpy(), cmap="gray")
axes[1].set_title("Preprocessed (middle slice, channel 0)")
for ax in axes:
    ax.axis("off")
plt.show()

Load the model. Pretrained CNN/ViT weights are optional here (download from the
[MRI-PTPCa repo](https://github.com/StandWisdom/MRI-based-Predicted-Transformer-for-Prostate-cancer/tree/main/pretrained_weights)
and point the paths below at them) — without them, the CNN backbones fall back to randomly
initialized MobileNetV3, which is structurally fine for running this pipeline but won't produce
meaningful embeddings.

In [ ]:
T2_MODEL_PATH = None
ADC_MODEL_PATH = None
VIT_MODEL_PATH = None

mri_model = load_mri_ptpca_model(
    t2_model_path=T2_MODEL_PATH,
    adc_model_path=ADC_MODEL_PATH,
    vit_model_path=VIT_MODEL_PATH,
    device=DEVICE,
)
if not any([T2_MODEL_PATH, ADC_MODEL_PATH, VIT_MODEL_PATH]):
    print("No pretrained weights provided -- using a randomly initialized CNN backbone.")

In [ ]:
adc_tensor = preprocess_mri_scan(sample_adc_path, sample_mask_path)
dwi_tensor = preprocess_mri_scan(sample_hbv_path, sample_mask_path) if sample_hbv_path.exists() else None
sample_mri_embedding = extract_mri_embedding(mri_model, t2_tensor, adc_tensor, dwi_tensor, device=DEVICE)
print(f"{sample_mri_id}: embedding shape {tuple(sample_mri_embedding.shape)}")
assert sample_mri_embedding.shape == (MRI_EMBEDDING_DIM,)

Now run it over every patient with both a T2W and an ADC series:

In [ ]:
t2_paths = sorted(MRI_IMAGE_DIR.glob("*/*_t2w.mha"))

for t2_path in tqdm(t2_paths, desc="MRI feature extraction"):
    mri_id = t2_path.stem.removesuffix("_t2w")
    out_path = MRI_FEATURE_DIR / f"{mri_id}_features.npy"
    if out_path.exists():
        continue

    adc_path = t2_path.with_name(f"{mri_id}_adc.mha")
    hbv_path = t2_path.with_name(f"{mri_id}_hbv.mha")
    mask_path = t2_path.with_name(f"{mri_id}_mask.mha")
    if not adc_path.exists():
        print(f"Skipping {mri_id}: no matching ADC series.")
        continue

    try:
        mask = mask_path if mask_path.exists() else None
        t2_t = preprocess_mri_scan(t2_path, mask)
        adc_t = preprocess_mri_scan(adc_path, mask)
        dwi_t = preprocess_mri_scan(hbv_path, mask) if hbv_path.exists() else None
        if dwi_t is None:
            print(f"{mri_id}: no HBV series -- zero-filling the third (DWI) input channel.")
        embedding = extract_mri_embedding(mri_model, t2_t, adc_t, dwi_t, device=DEVICE)
        np.save(out_path, embedding.cpu().numpy())
    except Exception as e:
        print(f"Failed on {mri_id}: {e}")

print(f"MRI features saved to {MRI_FEATURE_DIR}")

## Summary

This notebook produced:

| Artifact | Shape | Location |
|---|---|---|
| WSI patch embeddings | `(N patches, 1536)` per WSI | `data/features/wsi/*_features.npy` |
| MRI scan embeddings | `(2048,)` per scan | `data/features/mri/*_features.npy` |

Together with notebook 01's clinical embeddings (`data/prepared/clinical_embeddings/`, `(22,)`
each) and label CSVs (`data/labels/`), these are exactly the three feature dimensions the fusion
models in notebook 03 need: **WSI 1536** (patch-bag, pooled via attention-based MIL inside the
model), **MRI 2048**, **clinical 22**.

**Next up — `03_fusion_models.ipynb`:** early / intermediate / late fusion architectures that
combine these three modalities for both the classification and survival tasks.